In [6]:
import os
import json
import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Check if we are inside the 'notebooks' folder
if os.getcwd().endswith('notebooks'):
    os.chdir('..') # Go up one level to the project root

print(f"Current working directory: {os.getcwd()}")
print("Does 'data/inbox' exist?", os.path.exists("data/inbox"))
# 1. INITIALIZATION
spark = SparkSession.builder \
    .appName("NYC_Taxi_Incremental_ETL") \
    .config("spark.sql.shuffle.partitions", "2") \
    .getOrCreate()

INBOX = "data/inbox/"
OUTBOX = "data/outbox/trips_enriched.parquet"
SUMMARY_OUT = "data/outbox/daytype_summary.parquet"
LOOKUP = "data/inbox/taxi_zone_lookup.parquet"
MANIFEST = "state/manifest.json"

Current working directory: /home/jovyan
Does 'data/inbox' exist? True


In [7]:
# --- 2. INCREMENTAL INGESTION [cite: 15] ---
processed_files = []
if os.path.exists(MANIFEST):
    with open(MANIFEST, 'r') as f:
        manifest_data = json.load(f)
        processed_files = [entry['filename'] for entry in manifest_data]

all_files = [f for f in os.listdir(INBOX) if f.endswith('.parquet') and "lookup" not in f]
new_files = [f for f in all_files if f not in processed_files]

if not new_files:
    print("No new files. Summary will be recomputed from existing data.")
else:
    # Read only new files [cite: 17]
    raw_df = spark.read.parquet(*[os.path.join(INBOX, f) for f in new_files]) \
        .withColumn("source_file", F.input_file_name()) \
        .withColumn("ingested_at", F.current_timestamp())

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/home/jovyan/notebooks/data/inbox/yellow_tripdata_2025-01.parquet. SQLSTATE: 42K03

In [ ]:
# --- 3. TRANSFORMATIONS & CLEANING [cite: 22] ---
    # Rule: Filter invalid distances, passenger counts, and logical time errors 
    cleaned_df = raw_df.filter(
        (F.col("passenger_count") > 0) & 
        (F.col("trip_distance") > 0) &
        (F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
    )

    # Deduplication Key: Vendor + Timestamps + PU Location + Distance
    dedup_df = cleaned_df.dropDuplicates([
        "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", 
        "PULocationID", "trip_distance"
    ])

    # Derived Fields [cite: 37]
    transformed_df = dedup_df.withColumn(
        "trip_duration_minutes", 
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
    ).withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))

In [ ]:
# --- 4. SCENARIO LOGIC ---
    # Add day_type (1=Sunday, 7=Saturday)
    final_transformed_df = transformed_df.withColumn(
        "day_type",
        F.when(F.dayofweek("tpep_pickup_datetime").isin(1, 7), "Weekend").otherwise("Weekday")
    ).withColumn("pickup_year_month", F.date_format("tpep_pickup_datetime", "yyyy-MM"))

In [ ]:
# --- 5. ENRICHMENT [cite: 26] ---
    # Optimization: Broadcast Join for the small lookup table [cite: 50]
    zones_df = spark.read.parquet(LOOKUP).select("LocationID", "Zone")
    
    enriched_df = final_transformed_df.join(
        F.broadcast(zones_df), 
        final_transformed_df.PULocationID == zones_df.LocationID, 
        "left"
    ).withColumnRenamed("Zone", "pickup_zone_name").drop("LocationID")

In [8]:
# --- 6. IDEMPOTENT OUTPUT [cite: 21, 28] ---
    if os.path.exists(OUTBOX):
        existing_df = spark.read.parquet(OUTBOX)
        full_output_df = existing_df.unionByName(enriched_df, allowMissingColumns=True) \
            .dropDuplicates(["VendorID", "tpep_pickup_datetime", "PULocationID", "trip_distance"])
    else:
        full_output_df = enriched_df

    full_output_df.write.mode("overwrite").parquet(OUTBOX)

    # Update Manifest [cite: 19]
    new_entries = [{"filename": f, "ingested_at": str(datetime.datetime.now())} for f in new_files]
    with open(MANIFEST, 'w') as f:
        json.dump(processed_files + new_entries, f)

IndentationError: unexpected indent (2067497808.py, line 2)

In [9]:
# --- 7. SCENARIO SUMMARY (Recomputed from full output) ---
if os.path.exists(OUTBOX):
    full_data = spark.read.parquet(OUTBOX)
    summary_df = full_data.groupBy("day_type", "pickup_year_month") \
        .agg(
            F.count("*").alias("trip_count"),
            F.avg("trip_duration_minutes").alias("avg_trip_duration_minutes"),
            F.avg("fare_amount").alias("avg_fare_amount")
        )
    summary_df.write.mode("overwrite").parquet(SUMMARY_OUT)